# Laboratorio 02 - Búsqueda Adversaria (Min-Max y Alpha-Beta)

## I. DESARROLLANDO: BÚSQUEDA CON ADVERSARIO
### A. MODELAR BÚSQUEDA ADVERSARIA

In [1]:
class Game:
    def __init__(self, initial_state, players):
        self.initial_state = initial_state
        self.players = players

    def player(self, state):
        pass

    def opponent(self, player):
        pass

    def is_terminal(self, state):
        return False

    def result_actions(self, state):
        return set()

    def utility(self, state, player):
        return 0.0

### B. IMPLEMENTAR Min-Max

In [2]:
# ============================================================
# 1. Implemente la función para obtener el valor de un estado para MAX
# ============================================================
class MaxMinSearch:

    @staticmethod
    def max_value(game, state):
        if game.is_terminal(state):
            return game.utility(state, game.player(state))
        best_value = float('-inf')
        for successor in game.result_actions(state):
            value = MaxMinSearch.min_value(game, successor)
            best_value = max(best_value, value)
        return best_value

    @staticmethod
    def min_value(game, state):
        if game.is_terminal(state):
            return game.utility(state, game.player(state))
        worst_value = float('inf')
        for successor in game.result_actions(state):
            value = MaxMinSearch.max_value(game, successor)
            worst_value = min(worst_value, value)
        return worst_value

    @staticmethod
    def minmax_search(game, state):
        if game.player(state) == game.players[0]:
            return MaxMinSearch.max_value(game, state)
        else:
            return MaxMinSearch.min_value(game, state)
# ============================================================
# 2. Implemente la función para obtener el valor de un estado para MIN
# ============================================================
# ============================================================
# 3. Implemente el algoritmo de búsqueda del Min-Max
# ============================================================

In [3]:
from collections import deque

# ============================================================
# 1. Implemente la función que juega en base a un jugador dado (MAX o MIN)
# ============================================================
def play_game(game, player):
    state = game.initial_state
    steps = [state]

    while not game.is_terminal(state):
        shift = game.player(state)
        best_successor = None
        best_value = float('-inf') if shift == game.players[0] else float('inf')
        for successor in game.result_actions(state):
            value = MaxMinSearch.minmax_search(game, successor)
            if (shift == game.players[0] and value > best_value) or (shift == game.players[1] and value < best_value):
                best_value = value
                best_successor = successor
        state = best_successor
        steps.append(state)
    return steps
# ============================================================
# 2. Represente el espacio de búsqueda de ejemplo
# use como referencia el modelado del punto A
# ============================================================
class TreeGame(Game):
    def __init__(self, initial_state, players, nodes):
        super().__init__(initial_state, players)
        self.nodes = nodes

    def is_terminal(self, state):
        return isinstance(state, (int, float))

    def depth(self, state):
        queue = deque()
        queue.append((self.initial_state, 0))
        while queue:
            node, node_depth = queue.popleft()
            if node == state:
                return node_depth
            if self.is_terminal(node):
                continue
            for successor in self.nodes[node]:
                queue.append((successor, node_depth + 1))
        return -1

    def player(self, state):
        if self.depth(state) % 2 == 0:
            return self.players[0]
        else:
            return self.players[1]

    def opponent(self, player):
        if player == self.players[0]:
            return self.players[1]
        else:
            return self.players[0]

    def result_actions(self, state):
        if self.is_terminal(state):
            return set()
        else:
            return set(self.nodes[state])

    def utility(self, state, player):
        return float(state)
# ============================================================
# 3. Use la implementación del algoritmo de búsqueda min-max
# aplicándola al estado inicial del espacio de búsqueda ejemplo
# ============================================================
nodes = {
    "A": ["B", "C", "D"],
    "B": ["E", "F", "G"],
    "C": ["H", "I", "J"],
    "D": ["K", "L", "M"],
    "E": [8, 7, 2],
    "F": [9, 1, 6],
    "G": [2, 4, 1],
    "H": [1, 3, 5],
    "I": [3, 9, 2],
    "J": [6, 5, 2],
    "K": [1, 2, 3],
    "L": [9, 7, 2],
    "M": [16, 6, 4],
}
# ============================================================
# 4. Use la función `play_game` para jugar como MAX
# mostrando la secuencia de acciones y la utilidad final
# ============================================================
game = TreeGame(
    initial_state="A",
    players=("J1", "J2"),
    nodes=nodes
)

print("Secuencia de play_game:", play_game(game, "J1"))

Secuencia de play_game: ['A', 'C', 'H', 5]


### Implemente el algoritmo de búsqueda adversaria Alpha-Beta

In [4]:
# ============================================================
# 1. Haga las modificaciones a la función del jugador MAX
# ============================================================
def max_value(game, state, alpha, beta):
    if game.is_terminal(state):
        return game.utility(state, game.player(state))
    value = float('-inf')
    for successor in game.result_actions(state):
        value = max(value, min_value(game, successor, alpha, beta))
        if value >= beta:
            return value
        alpha = max(alpha, value)
    return value

In [5]:
# ============================================================
# 2. Haga las modificaciones a la función del jugador MIN
# ============================================================
def min_value(game, state, alpha, beta):
    if game.is_terminal(state):
        return game.utility(state, game.player(state))
    value = float('inf')
    for successor in game.result_actions(state):
        value = min(value, max_value(game, successor, alpha, beta))
        if value <= alpha:
            return value
        beta = min(beta, value)
    return value

In [6]:
# ============================================================
# 3. Implemente el algoritmo de búsqueda alpha-beta
# ============================================================
def alpha_beta_search(game, state):
    if game.player(state) == game.players[0]:
        return max_value(game, state, float('-inf'), float('inf'))
    else:
        return min_value(game, state, float('-inf'), float('inf'))

In [7]:
# ============================================================
# 4. Usando la representación del problema de ejemplo previa,
# pruebe el algoritmo alpha-beta en el espacio de ejemplo
# ============================================================
print("Utilidad raiz con Alpha-Beta:", alpha_beta_search(game, "A"))

Utilidad raiz con Alpha-Beta: 5.0


## II. SOLUCIONAR PROBLEMAS
### A. SOLUCIONANDO — A.1 Tic-Tac-Toe

In [8]:
# ============================================================
# 1. Represente el diseño del problema seleccionado
# ============================================================
class TicTacToeGame(Game):
    LINES = [
        (0, 1, 2), (3, 4, 5), (6, 7, 8),  # filas
        (0, 3, 6), (1, 4, 7), (2, 5, 8),  # columnas
        (0, 4, 8), (2, 4, 6),             # diagonales
    ]

    def __init__(self):
        super().__init__(initial_state=tuple(' ' for _ in range(9)), players=('X', 'O'))

    def player(self, state):
        x_count = state.count('X')
        o_count = state.count('O')
        return 'X' if x_count == o_count else 'O'

    def opponent(self, player):
        return 'O' if player == 'X' else 'X'

    def winner(self, state):
        for a, b, c in self.LINES:
            if state[a] != ' ' and state[a] == state[b] == state[c]:
                return state[a]
        return None

    def is_terminal(self, state):
        return self.winner(state) is not None or ' ' not in state

In [9]:
# ============================================================
# 2. Agregue lo necesario a su representación del problema
# ============================================================
def result_actions(self, state):
    if self.is_terminal(state):
        return set()
    mark = self.player(state)
    successors = set()
    for i in range(9):
        if state[i] == ' ':
            successors.add(state[:i] + (mark,) + state[i + 1:])
    return successors

def utility(self, state, player):
    w = self.winner(state)
    if w is None:
        return 0.0
    return 1.0 if w == 'X' else -1.0

TicTacToeGame.result_actions = result_actions
TicTacToeGame.utility = utility

In [10]:
# ============================================================
# 3. Use el algoritmo alpha-beta aplicándola al problema de seleccionado
# ============================================================
ttt = TicTacToeGame()
print("Utilidad del estado inicial (Alpha-Beta):", alpha_beta_search(ttt, ttt.initial_state))

Utilidad del estado inicial (Alpha-Beta): 0.0


In [11]:
# ============================================================
# 4. Use la función para jugar como MAX
# mostrando la secuencia de acciones y la utilidad final
# ============================================================
def play_game_ab(game, player):
    state = game.initial_state
    steps = [state]
    while not game.is_terminal(state):
        shift = game.player(state)
        best_successor = None
        best_value = float('-inf') if shift == game.players[0] else float('inf')
        for successor in game.result_actions(state):
            value = alpha_beta_search(game, successor)
            if (shift == game.players[0] and value > best_value) or (shift == game.players[1] and value < best_value):
                best_value = value
                best_successor = successor
        state = best_successor
        steps.append(state)
    return steps

def print_board(state):
    rows = [' | '.join(state[r * 3:r * 3 + 3]) for r in range(3)]
    print(('\n' + '-' * 9 + '\n').join(rows))

ttt_sequence = play_game_ab(ttt, 'X')
print("Numero de estados en la secuencia:", len(ttt_sequence))
print("Utilidad final:", ttt.utility(ttt_sequence[-1], 'X'))
print("Tablero final:")
print_board(ttt_sequence[-1])

Numero de estados en la secuencia: 10
Utilidad final: 0.0
Tablero final:
X | X | O
---------
O | X | X
---------
X | O | O
